In [ ]:
from plotnine import *
import pandas as pd

# Load all benchmark data
## Convergence benchmark data

In [ ]:
def process_ebe_data(name):
    df_raw = pd.read_csv("outputs/ebe_pysaem_" + name + ".csv")
    ebe_pysaem = df_raw
    ebe_pysaem["algo"] = "pysaem"
    ebe_pysaem["setting"] = name
    ebe_nlmixr = pd.read_csv("outputs/ebe_nlmixr_" + name + ".csv")
    ebe_nlmixr["algo"] = "nlmixr"
    ebe_nlmixr["setting"] = name
    ebe = pd.concat([ebe_pysaem, ebe_nlmixr])
    descriptors = ["k_eL", "R0", "Vc"]

    ebe = ebe.melt(
        id_vars=["id", "algo", "setting"], value_vars=descriptors, var_name="descriptor"
    ).reset_index()

    return ebe


def process_ebe_true(name):
    df_raw = pd.read_csv(f"data/synthetic_data_50pts_{name}.csv")
    ebe_true = (
        df_raw[["id", "protocol_arm", "true_k_eL", "true_R0", "true_Vc"]]
        .drop_duplicates()
        .rename(columns={"true_k_eL": "k_eL", "true_R0": "R0", "true_Vc": "Vc"})
        .melt(
            id_vars=["id", "protocol_arm"],
            value_vars=["k_eL", "R0", "Vc"],
            var_name="descriptor",
            value_name="true_value",
        )
    ).assign(setting=name)
    obs_df = df_raw[["id", "time", "protocol_arm", "DV"]]
    obs_df = obs_df.loc[~obs_df["DV"].isna()]
    obs_df["setting"] = name

    return ebe_true, obs_df


ebe_obs_1d = process_ebe_data("1_dose")
ebe_obs_2d = process_ebe_data("2_dose")
ebe_obs = pd.concat([ebe_obs_1d, ebe_obs_2d])

ebe_true_2d, obs_df_1d = process_ebe_true("2_dose")
ebe_true_1d, obs_df_2d = process_ebe_true("1_dose")
ebe_true = pd.concat([ebe_true_1d, ebe_true_2d])

comp_df = pd.concat(
    [ebe_obs, ebe_true.rename(columns={"true_value": "value"}).assign(algo="true")]
)
display(comp_df.head())

obs_df = pd.concat([obs_df_1d, obs_df_2d])
display(obs_df)

## Performance benchmark data

In [ ]:
perf_nlmixr_raw = pd.read_csv("outputs/performance_nlmixr.csv")
perf_pysaem_raw = pd.read_csv("outputs/performance_pysaem.csv")

perf_nlmixr = perf_nlmixr_raw
perf_nlmixr["algo"] = "nlmixr"

perf_pysaem_full = perf_pysaem_raw[["time", "nb_patients"]]
perf_pysaem_full["algo"] = "pysaem"

full_df = pd.concat([perf_nlmixr, perf_pysaem_full])

decomp_pysaem = perf_pysaem_raw[["time_saem", "time_train", "nb_patients"]].assign(
    algo="pysaem"
)
decomp_nlmixr = perf_nlmixr_raw.assign(algo="nlmixr").rename(
    columns={"time": "time_saem"}
)
decomp_df = (
    pd.concat([decomp_pysaem, decomp_nlmixr])
    .rename(columns={"time_saem": ": SAEM", "time_train": ": Surrogate training"})
    .melt(
        id_vars=["nb_patients", "algo"],
        var_name="part",
        value_vars=[": SAEM", ": Surrogate training"],
        value_name="time",
    )
    .reset_index(drop=True)
    .groupby(["nb_patients", "algo", "part"])
    .agg({"time": "mean"})
    .reset_index()
)
decomp_df["color"] = decomp_df.apply(lambda row: row["algo"] + row["part"], axis=1)
decomp_df = decomp_df.loc[~decomp_df["time"].isna()]
display(decomp_df)

In [ ]:
final_fig_size = (12, 10)
color_scale = "Accent"

In [ ]:
p1 = (
    ggplot(obs_df, aes(x="time", y="DV", group="id"))
    + geom_line(alpha=0.2)
    + geom_point(alpha=0.2)
    + facet_grid(
        rows="setting",
        cols="protocol_arm",
        labeller={
            "1_dose": "Single dose",
            "2_dose": "Two doses",
            "arm_1": "Dose 100 nmol",
            "arm_2": "Dose 10 nmol",
        },
    )
    + theme_minimal()
    + theme(
        figure_size=final_fig_size,
        plot_background=element_rect(fill="none"),
    )
    + labs(
        x="Time (h)",
        y="DV (log)",
        subtitle="(A) Synthetic data for convergence benchmark",
    )
)
p1

In [ ]:
p2 = (
    ggplot(comp_df, aes(x="value", fill="algo", color="algo"))
    + geom_density(alpha=0.75)
    + facet_grid(
        cols="descriptor",
        rows="setting",
        scales="free",
        labeller={"1_dose": "Single dose", "2_dose": "Two doses"},
    )
    + scale_x_continuous(trans="log10")
    + theme_minimal()
    + labs(
        x="Parameter value",
        y="Distribution density",
        color="Source",
        fill="Source",
        subtitle="(B) Individual parameter estimates for TMDD model",
    )
    + scale_fill_cmap_d(color_scale)
    + scale_color_cmap_d(color_scale)
    + theme(
        figure_size=final_fig_size,
        plot_background=element_rect(fill="none"),
        legend_position="bottom",
    )
)
p2

In [ ]:
p3 = (
    ggplot(
        full_df,
        aes(x="nb_patients", y="time/nb_patients", color="algo"),
    )
    + geom_point(alpha=0.5)
    + geom_smooth(method="lm")
    + scale_x_continuous(trans="log10")
    + scale_y_continuous(trans="log10")
    + scale_color_cmap_d(color_scale)
    + theme_minimal()
    + theme(
        legend_position="bottom",
        figure_size=final_fig_size,
        plot_background=element_rect(fill="none"),
    )
    + labs(
        subtitle="(C) Per patient estimation time for SAEM on TMDD model\n100/100 iterations, 1 MCMC, 5x repeat",
        x="Data set size (patient count)",
        y="Estimation time per patient (s)",
        color="Implementation",
    )
)
p3

In [ ]:
p4 = (
    ggplot(decomp_df, aes(x="algo", y="time", fill="color"))
    + geom_bar(stat="identity", position="stack")
    + facet_grid(cols="nb_patients", labeller=labeller(cols=(lambda n: f"{n} pts")))
    + scale_fill_cmap_d(color_scale)
    + theme_minimal()
    + theme(
        figure_size=final_fig_size,
        axis_text_x=element_blank(),
        legend_position="bottom",
        plot_background=element_rect(fill="none"),
    )
    + labs(
        x="", y="Average time (s)", fill="", subtitle="(D) Average runtime decomposed"
    )
)
p4

In [ ]:
final_plot = (p1 | p2) / (p3 | p4)
final_plot.save("outputs/tmdd_benchmark.png", dpi=300)
final_plot.draw()